### Classical Preparation

For a function ( f: {0,1} → {0,1} ), there are four possible cases:

* **Constant:**
  ( f(0)=0, f(1)=0 )
  ( f(0)=1, f(1)=1 )

* **Balanced:**
  ( f(0)=0, f(1)=1 )
  ( f(0)=1, f(1)=0 )

A classical computer requires **2 queries** because evaluating only one input does not reveal whether the function gives the same or different output for the other input. Both ( f(0) ) and ( f(1) ) must be checked to determine if the function is constant or balanced.


In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer

In [16]:
def constant_oracle():
    qc = QuantumCircuit(2)
    return qc

In [17]:
def balanced_oracle():
    qc = QuantumCircuit(2)
    qc.cx(0, 1)
    return qc

In [26]:
def deutsch_circuit(oracle):
    qc = QuantumCircuit(2, 1)   # 2 qbits; q0 -> input, q1 -> ancilla & c1 -> store result.

    qc.x(1)     # Set ancilla qbit (q1) to state |1⟩ for phase.

    qc.h(0)     # q0 becomes 0 & 1 at once (SP).
    qc.h(1)     # q1 becomes special state for phase encoding.

    qc.compose(oracle, inplace=True) # Inserts a oracle into the circ -> Runs the function.
    # If output of function is diff. from input, it adds - (changing the phase).

    qc.h(0)     # Convert hidden phase info → measurable result.

    qc.measure(0, 0)

    sim = Aer.get_backend('aer_simulator')  # Get simulator.
    compiled = transpile(qc, sim)   # Prepare circ for simulator.

    result = sim.run(compiled, shots=1).result() # Run once as Deutsch algm. only needs 1 shot.

    counts = result.get_counts()    # Dict. like {'0': 1}

    bit = list(counts.keys())[0]    # To check if input qbit (q0) is constant OR changed.
    
    if (bit == "0"):
        print("The function is Constant.")
    else:
        print("The function is Balanced.")

In [27]:
print("Testing Constant Oracle:")
deutsch_circuit(constant_oracle())

Testing Constant Oracle:
The function is Constant.


In [28]:
print("Testing Balanced Oracle:")
deutsch_circuit(balanced_oracle())

Testing Balanced Oracle:
The function is Balanced.


The output is always the same because the oracle defines a fixed function,
& the Deutsch algorithm deterministically classifies that fixed function.